# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Logistic Regression first.** My Week-4 baseline is a hand-written rule built on one signal
(CTR-vs-position). This week's job is to replace it with a *learned* model that can combine
several signals (staleness, CTR gap, impressions, content_type, etc.) and weigh them
automatically — then compare precision@10 / precision@50 against the same baseline, same split.

Per `training-honest-models`: `is_declining_label` is a yes/no observed label → "Logistic
Regression, then Random Forest." I'm starting with LR specifically because:

- **Readable → stronger.** LR's coefficients are directly interpretable (this feature raises/
  lowers decline-probability by this much), so if a Random Forest later "wins," I'll know whether
  it earns that with real nonlinear structure or is just overfitting noise on 30k rows.
- **If LR already beats the rule baseline, that's "done"** — no need to reach for the opaque
  model just because it exists (simplicity is a feature, per the skill).
- **LR forces me to handle missingness/scaling explicitly** (no free lunch with NaNs the way
  trees sometimes tolerate), which fits the data skill's warning that missingness follows
  `content_type` — I have to make deliberate `has_*` flags rather than let a model silently
  split around gaps.
- Both LR and RF output a probability I can rank by, so either satisfies "ranking needs scores,
  not labels" for precision@K.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Same label construction as the Week-4 baseline notebook.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()

# Grouped by client_id, not random: random would let a client's rows appear in both
# train and test, so the model could "cheat" by learning that client's quirks instead
# of a generalizable decline signal. This is a single-snapshot dataset (trailing-90-day
# metrics, no repeated timestamps per row), so there's no time axis to split on instead
# -- client_id is the only honest grouping key, matching the baseline's own use of
# client_id for grouping (never as a feature).
#
# test_size raised to 0.3 (from an initial 0.2): with only ~32 clients total, a 0.2 test
# split landed just 7 clients in test and a decline rate (0.511) close to, but not
# exactly matching, the full base rate (0.542) -- small-group splits carry real
# seed-dependent variance. A larger held-out slice gives a steadier precision@K estimate
# without starving the model of training clients.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, df["is_declining_label"], groups=df["client_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(train_df):,} rows, {train_df['client_id'].nunique()} clients, "
      f"decline rate {train_df['is_declining_label'].mean():.3f}")
print(f"test:  {len(test_df):,} rows, {test_df['client_id'].nunique()} clients, "
      f"decline rate {test_df['is_declining_label'].mean():.3f}")
print(f"full dataset decline rate (base rate): {base_rate:.3f}")

train: 19,166 rows, 22 clients, decline rate 0.532
test:  10,834 rows, 10 clients, decline rate 0.559
full dataset decline rate (base rate): 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# avg_position == 0 means "no position data", not rank zero (data skill gotcha) --
# convert to NaN so it's imputed/flagged like any other missing value, not treated as a
# real (excellent) rank.
for frame in (train_df, test_df):
    frame["avg_position"] = frame["avg_position"].replace(0, np.nan)

LABEL_DERIVED = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}
# LEAKAGE CHECK: (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d
# correlates with trend_pct at 1.000 -- trend_pct is effectively COMPUTED from the
# last_30d/prev_30d impressions pair, so *_last_30d columns reconstruct the label almost
# exactly (a first pass with them included hit precision@50 = 1.00, the skill's own
# "suspiciously perfect = probably leakage" tell). Per the data skill's window-alignment
# rule ("if your label lives in the last 30 days, only *_prev30-style columns are safe"),
# the *_last_30d trio is dropped; *_prev_30d is kept -- it describes the period BEFORE the
# comparison window and can't by itself reconstruct trend direction.
LEAKY_WINDOW_COLS = {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d"}
DROP_COLS = LABEL_DERIVED | ID_COLS | LEAKY_WINDOW_COLS

NUMERIC_FEATURES = [
    c for c in train_df.select_dtypes(include="number").columns if c not in DROP_COLS
]
CATEGORICAL_FEATURES = [
    c for c in train_df.select_dtypes(include="object").columns if c not in DROP_COLS
]

print(f"{len(NUMERIC_FEATURES)} numeric features, {len(CATEGORICAL_FEATURES)} categorical features")
print("numeric:", NUMERIC_FEATURES)
print("categorical:", CATEGORICAL_FEATURES)

X_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train_df["is_declining_label"]
X_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = test_df["is_declining_label"]

# Missingness follows content_type (data skill gotcha) -- a blind fillna(0) would inject
# a fake category signal (e.g. word_count=0 looking like "no words" instead of "not
# recorded for this content_type"). SimpleImputer(add_indicator=True) fills numeric NaNs
# with the median AND adds an explicit missing-flag column per feature, which is the
# has_* pattern the skill calls for -- done through the pipeline instead of by hand.
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]
print("\nLR fit on train, scored test set. Sample of model_score:")
print(test_df["model_score"].describe())

26 numeric features, 11 categorical features
numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']



LR fit on train, scored test set. Sample of model_score:
count    1.083400e+04
mean     6.395209e-01
std      1.844472e-01
min      2.694866e-18
25%      5.459759e-01
50%      6.805997e-01
75%      7.718877e-01
max      1.000000e+00
Name: model_score, dtype: float64


In [3]:
VISIBLE_MIN_IMPRESSIONS = 500  # same threshold as the Week-4 baseline

# Re-run the EXACT Week-4 rule logic, but restricted to test_df only -- "same split as
# the baseline" means the baseline has to be scored on this same held-out slice, not on
# the full 30,000-row set it was originally evaluated against.
has_position_test = test_df["avg_position"].notna()
visible_test = test_df["impressions_90d"] >= VISIBLE_MIN_IMPRESSIONS
tier_median_ctr_test = pd.Series(np.nan, index=test_df.index)
tier_median_ctr_test.loc[has_position_test] = (
    test_df.loc[has_position_test].groupby("position_tier")["ctr"].transform("median")
)
ctr_underperforms_test = pd.Series(False, index=test_df.index)
ctr_underperforms_test.loc[has_position_test] = (
    test_df.loc[has_position_test, "ctr"] < tier_median_ctr_test.loc[has_position_test]
)
flagged_test = has_position_test & visible_test & ctr_underperforms_test
test_df["rule_score"] = np.where(flagged_test, tier_median_ctr_test - test_df["ctr"], 0)


def precision_at_k(labels: pd.Series, k: int) -> float:
    return labels.head(k).mean()


rule_ranked = test_df.sort_values(
    ["rule_score", "impressions_90d"], ascending=False
).reset_index(drop=True)
model_ranked = test_df.sort_values("model_score", ascending=False).reset_index(drop=True)

test_base_rate = y_test.mean()
rows = []
for k in (10, 50):
    rows.append({
        "method": "baseline rule (v2, CTR-gap)",
        "k": k,
        "precision@k": precision_at_k(rule_ranked["is_declining_label"], k),
    })
    rows.append({
        "method": "logistic regression",
        "k": k,
        "precision@k": precision_at_k(model_ranked["is_declining_label"], k),
    })

comparison = pd.DataFrame(rows)
comparison["base_rate (test)"] = test_base_rate
print(f"test set base rate: {test_base_rate:.3f}  ({len(test_df):,} rows, "
      f"{test_df['client_id'].nunique()} clients)\n")
print(comparison.to_string(index=False))

test set base rate: 0.559  (10,834 rows, 10 clients)

                     method  k  precision@k  base_rate (test)
baseline rule (v2, CTR-gap) 10         0.90          0.559442
        logistic regression 10         0.80          0.559442
baseline rule (v2, CTR-gap) 50         0.72          0.559442
        logistic regression 50         0.74          0.559442


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# What the LR leans on: pull feature names out of the fitted ColumnTransformer and pair
# them with the logreg coefficients. Positive coefficient = pushes decline-probability UP.
feature_names = model.named_steps["preprocess"].get_feature_names_out()
coefs = model.named_steps["logreg"].coef_[0]

coef_table = (
    pd.DataFrame({"feature": feature_names, "coef": coefs})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .drop(columns="abs_coef")
)
print("Top 15 features by |coefficient| (LR, standardized inputs):")
print(coef_table.head(15).to_string(index=False))

Top 15 features by |coefficient| (LR, standardized inputs):
                               feature      coef
             num__impressions_prev_30d  2.980925
                  num__impressions_90d -2.759918
                        num__users_90d -1.860428
    num__missingindicator_avg_position -1.094519
                    num__pageviews_90d  0.936853
                     num__sessions_90d  0.827163
            num__days_with_impressions  0.771183
            cat__model_used_gpt-5-mini  0.760493
             cat__freshness_tier_31-90 -0.684041
                 num__content_age_days -0.588318
      cat__model_used_gemini-2.5-flash -0.565472
        cat__competition_level_missing  0.496169
                     num__avg_position -0.464876
cat__model_used_gemini-3-flash-preview -0.451834
               num__days_with_sessions -0.403722


In [5]:
# Wrong cases: the model's most confident picks (top 50 by model_score) that did NOT
# actually decline. precision@50 = 0.74, so 13 of these -- enough to look for a shared
# pattern instead of guessing from 1-2 rows.
top50 = model_ranked.head(50).copy()
wrong = top50[top50["is_declining_label"] == 0]
print(f"{len(wrong)} of the top 50 by model_score were NOT declining "
      f"({len(wrong)/50:.0%} of the top 50, vs test base rate {test_base_rate:.1%}).\n")

WATCH_COLS = [
    "content_type", "freshness_tier", "impressions_prev_30d", "impressions_90d",
    "avg_position", "position_tier", "ctr", "main_intent",
]
print(wrong[["content_id", "model_score"] + WATCH_COLS].to_string(index=False))

print("\nShared traits across the wrong picks, vs the full top 50:")
for col in ["content_type", "freshness_tier", "position_tier", "main_intent"]:
    print(f"\n{col} -- wrong picks:")
    print(wrong[col].value_counts(normalize=True).round(2))
    print(f"{col} -- all top 50:")
    print(top50[col].value_counts(normalize=True).round(2))

13 of the top 50 by model_score were NOT declining (26% of the top 50, vs test base rate 55.9%).

          content_id  model_score    content_type freshness_tier  impressions_prev_30d  impressions_90d  avg_position position_tier  ctr   main_intent
content_73c54f78c06a     0.999989 keyword article           0-30                 97200           213963           4.7        page_1 0.10 informational
content_f3e5505473ad     0.999934 keyword article         91-180                 30551            89750           9.4        page_1 1.57 informational
content_d0513fb2a904     0.999690 keyword article         91-180                  1265             4293          21.4      page_3_5 0.33 informational
content_2db251d1a841     0.999583 keyword article           0-30                 84550           198671           5.6        page_1 0.18 informational
content_b28d1efd668f     0.999426 keyword article         91-180                110679           286608          26.2      page_3_5 0.06 transactio

**What the model leans on.** The two largest LR coefficients are `impressions_prev_30d`
(+2.98) and `impressions_90d` (−2.76) — both raw traffic-volume measures, not severity
measures. They're correlated (`impressions_90d` roughly contains `impressions_prev_30d`), so
LR is likely splitting one real "how big is this page" signal into two coefficients that
partly cancel — worth treating as one blurry volume signal, not two independent ones. Other
features (`missingindicator_avg_position`, `days_with_impressions`, `freshness_tier_31-90`)
make more direct sense and are smaller in magnitude.

**Where the model is wrong.** Of the top 50 rows by `model_score`, 13 (26%) were not actually
declining — but they aren't scattered mistakes. Every one of those 13 sits at **0.99+
confidence**, and all 13 are `keyword article` pages with large `impressions_prev_30d` /
`impressions_90d` values (page_1 or page_3_5 position tier, mostly informational intent).

**This is the same mistake my v1 baseline made, just automated.** My Week-4 baseline's first
attempt ranked flagged pages by raw `impressions_90d` ("more traffic = bigger opportunity")
and scored *below* the base rate — because sorting by size pulled big, structurally low-risk
pages ahead of smaller pages with a genuinely worse CTR problem. I fixed that by ranking by
`ctr_gap` (severity) instead. The LR never got that correction: with `impressions_prev_30d` /
`impressions_90d` as its two biggest coefficients, it's effectively treating "this is a huge
page" as evidence of decline risk, when size alone isn't what the rule-tuning process showed
to be predictive — severity relative to a page's own position tier is. That confusion is
concentrated at the very top of the model's ranking (maxed-out 0.99+ scores), which is exactly
where precision@K weights errors most heavily.

**Net read:** LR still clears the base rate comfortably (0.74 vs 0.56 at k=50) and edges the
rule at k=50, but the rule is still sharper at k=10 (0.90 vs 0.80) precisely because the rule's
`ctr_gap` signal doesn't have this volume-confusion failure mode. A natural next iteration —
not attempted here — would be adding `ctr_gap` itself (or a position-relative CTR feature) as
an LR input, so the model gets access to the severity signal that already worked for the rule,
rather than relying so heavily on raw traffic size.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.